In [0]:
from calc_functions import *

In [0]:
%sql
DROP TABLE IF EXISTS dev.mohit_gangwani.ad_viewing_type_dma_overall;
CREATE TABLE dev.mohit_gangwani.ad_viewing_type_dma_overall AS
SELECT vc.external_id AS ad_id
, CASE WHEN st.station_id IS NOT NULL THEN st.local_or_national
       WHEN inp.reported_input_source = 'ANTENNA' THEN 'Local'
       WHEN NVL(vc.prev_vizio_epg_station, vc.next_vizio_epg_station) IS NOT NULL THEN 'APPS'
       WHEN inp.reported_input_source = 'APPS' THEN 'APPS'
       ELSE 'Unknown' END AS viewing_type
, vc.fk_dma_id
, COUNT(DISTINCT vc.fk_tvid) AS tv_count
, COUNT(DISTINCT vc.fk_tvid||'_'||vc.session_start) AS impression_count
FROM prod.detection.viewing_commercials_firehose_dedup vc
LEFT JOIN prod.detection.epg_station st
  ON st.station_id = NVL(vc.prev_station_id, vc.next_station_id)
 AND st.vendor_name = 'TIVO'
JOIN prod.detection.commercial_id_external_firehose cief
  ON cief.external_id = vc.external_id
JOIN prod.detection.clients cl
  ON cl.client_id = cief.fk_client_id
LEFT JOIN prod.detection.input_source inp
  ON inp.input_source_id = vc.fk_input_source_id
WHERE vc.session_start >= CURRENT_DATE - 7
  AND vc.session_start < CURRENT_DATE
  AND vc.fk_zoo_id = 17
  AND cl.client_name = 'kinetiq'
  AND vc.fk_dma_id IS NOT NULL
GROUP BY 1, 2, 3
HAVING tv_count >= 10
   AND impression_count >= 20;

In [0]:
%sql
DROP TABLE IF EXISTS dev.mohit_gangwani.ad_viewing_type_dmas_count;
CREATE TABLE dev.mohit_gangwani.ad_viewing_type_dmas_count AS
WITH ad_dma AS (
  SELECT a.ad_id, a.viewing_type, a.fk_dma_id, a.impression_count, a.tv_count
  FROM dev.mohit_gangwani.ad_viewing_type_dma_overall a
  GROUP BY ALL
)
SELECT a.ad_id
, a.viewing_type
, COUNT(DISTINCT a.fk_dma_id) AS dma_count
, SUM(a.impression_count) AS ttl_impressions
, SUM(a.tv_count) AS tv_count
, dma_count*1.0/ 210 AS dma_coverage
FROM ad_dma a
GROUP BY 1, 2;

In [0]:
%sql
SELECT CASE WHEN st.station_id IS NOT NULL THEN st.local_or_national
            WHEN inp.reported_input_source = 'ANTENNA' THEN 'Local'
            WHEN NVL(vc.prev_vizio_epg_station, vc.next_vizio_epg_station) IS NOT NULL THEN 'APPS'
            WHEN inp.reported_input_source = 'APPS' THEN 'APPS'
            ELSE 'Unknown' END AS viewing_type
, COUNT(DISTINCT vc.fk_tvid) AS tv_count
, COUNT(DISTINCT vc.fk_tvid||'_'||vc.session_start) AS impression_count
FROM prod.detection.viewing_commercials_firehose_dedup vc
LEFT JOIN prod.detection.epg_station st
  ON st.station_id = NVL(vc.prev_station_id, vc.next_station_id)
 AND st.vendor_name = 'TIVO'
JOIN prod.detection.commercial_id_external_firehose cief
  ON cief.external_id = vc.external_id
JOIN prod.detection.clients cl
  ON cl.client_id = cief.fk_client_id
LEFT JOIN prod.detection.input_source inp
  ON inp.input_source_id = vc.fk_input_source_id
WHERE vc.session_start >= CURRENT_DATE - 7
  AND vc.session_start < CURRENT_DATE
  AND vc.fk_zoo_id = 17
  AND cl.client_name = 'kinetiq'
  AND vc.fk_dma_id IS NOT NULL
GROUP BY 1

In [0]:
%sql
SELECT viewing_type
, APPROX_PERCENTILE(dma_count, 0.25) AS perc_25
, APPROX_PERCENTILE(dma_count, 0.50) AS perc_50
, APPROX_PERCENTILE(dma_count, 0.75) AS perc_75
, APPROX_PERCENTILE(dma_count, 0.90) AS perc_90
, AVG(dma_count) AS avg_dma_count
FROM dev.mohit_gangwani.ad_viewing_type_dmas_count
GROUP BY 1

In [0]:
%sql
SELECT viewing_type
, dma_count
, COUNT(DISTINCT ad_id)
FROM dev.mohit_gangwani.ad_viewing_type_dmas_count
GROUP BY 1, 2

In [0]:
df_from_sql = spark.sql(
    """WITH ovrl AS (   
        SELECT ad_id, SUM(impression_count) AS ttl_count
        FROM dev.mohit_gangwani.ad_viewing_type_dma_overall
        WHERE viewing_type != 'Unknown'
        GROUP BY 1
        )
        , ad_filter AS (
        SELECT ad_id, ttl_count, DENSE_RANK() OVER (ORDER BY ttl_count DESC) AS rk
        FROM ovrl
        )
        SELECT a.*
        FROM dev.mohit_gangwani.ad_viewing_type_dma_overall a
        JOIN ad_filter f
        ON f.ad_id = a.ad_id
        WHERE f.rk <= 10000
        GROUP BY ALL;
        """
)
vdf = df_from_sql.toPandas()
vdf.head(20)

In [0]:
v_agg_df = vdf.groupby(['ad_id', 'viewing_type']).agg(
    gini=('impression_count', lambda x: gini(x)),
    entropy=('impression_count', lambda x: locality_index(x))
).reset_index()

v_agg_df.head(10)

In [0]:
v_agg_df.describe()

In [0]:
sns.scatterplot(
    data=v_agg_df[v_agg_df.viewing_type != 'Unknown'], 
    x='gini', 
    y='entropy',
    hue='viewing_type',
    alpha=0.35, 
    s=30
)
plt.title("Ad Footprint Concentration by Viewing Type")
plt.xlabel("Normalized Gini (Inequality)")
plt.ylabel("Locality Index (Normalized Entropy)")
plt.show()

In [0]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.boxplot(data=v_agg_df, x='viewing_type', y='gini', ax=axes[0])
sns.boxplot(data=v_agg_df, x='viewing_type', y='entropy', ax=axes[1])
axes[0].set_title("Gini by Viewing Type")
axes[1].set_title("Entropy/Locality Index by Viewing Type")

In [0]:
ax = sns.kdeplot(
    data=v_agg_df[v_agg_df.viewing_type.isin(['APPS', 'National', 'Local'])],
    x='gini',
    y='entropy',
    hue='viewing_type',
    fill=True, alpha=0.25
)
sns.move_legend(ax, "upper left")
# plt.show()

In [0]:
summary = v_agg_df.groupby('viewing_type')[['gini','entropy']].median().reset_index()

sns.scatterplot(
    data=summary,
    x='gini',
    y='entropy',
    hue='viewing_type',
    s=200, edgecolor='black'
)
plt.title("Median Gini vs. Entropy by Station Type")